In [ ]:
# ruff: noqa: F401
import subprocess
from pathlib import Path

import board
from agent_framework import Agent, MCPStdioTool
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from mcp.client.stdio import StdioServerParameters, stdio_client
from rich.console import Console
from rich.markdown import Markdown

load_dotenv(override=True)
console = Console()

### Create and run Agent


In [2]:
MODEL = "gpt-5.4-nano"

client = OpenAIChatClient(model=MODEL)

agent = Agent(client, "You are an agent who provides answer in short sentences")

result = await agent.run("How is Agentic-AI beneficial for human life?")

console.print(Markdown(result.text))

Agentic AI can benefit human life in several ways:                                                                 

 • Automates repetitive work.                                                                                      
 • Helps people finish tasks faster.                                                                               
 • Provides 24/7 support for common questions.                                                                     
 • Assists in healthcare by summarizing records and flagging risk patterns.                                        
 • Supports education with personalized practice and tutoring.                                                     
 • Improves accessibility through adaptive help for disabilities.                                                  
 • Reduces errors in paperwork by checking consistency.                                                            
 • Speeds up research by exploring ideas and compiling findings.                                                   
 • Enhances safety by monitoring systems and alerting humans early.                                                
 • Improves logistics by optimizing routes and resource use.                                                       
 • Enables disaster response with faster planning and information gathering.                                       

Main benefits are speed, scale, and personalized assistance.

### Using tools


In [3]:
board.reset_board()
board.add_goal(
    """
    Read notes.txt, read its contents and convert into Bengali, and write to bengali.txt.
    Overwrite the file if it already exists.
    """
)
board.show_board()
board.list_todos()

Goal #1: 
    Read notes.txt, read its contents and convert into Bengali, and write to bengali.txt.
    Overwrite the file if it already exists.

[{'id': 1,
  'parent_id': None,
  'task': '\n    Read notes.txt, read its contents and convert into Bengali, and write to bengali.txt.\n    Overwrite the file if it already exists.\n    ',
  'status': 'pending',
  'result': ''}]

In [4]:
def show_todos() -> list[dict]:
    """List every tool in the todo list. A goal has a parent id is None. A step has the parent_id set to it's goal_id"""
    return board.list_todos()


def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps

    Args:
        goal_id: The id of the goal to decompose
        steps: Short description of the steps to be taken in order

    """
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}


def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (step or goal) as done and add a short summary

    Args:
        task_id: The id of the todo tak to be done
        result: A short summary of the task completed
    """
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "Done"}

In [5]:
board_agent = Agent(
    client, "You are an agent who can manage a todo board", tools=[plan_steps, show_todos, complete_task]
)

result = await board_agent.run(messages="What is on the board right now, and what is the status")

console.print(Markdown(result.text))

Right now the board has 1 item:                                                                                    

 • Goal #1 (parent: None): “Read notes.txt, read its contents and convert into Bengali, and write to bengali.txt.  
   Overwrite the file if it already exists.”                                                                       
    • Status: pending                                                                                              
    • Result: (empty)

### Using MCP


In [6]:
workspace = Path("workspace").resolve()


class FilesystemMCP(MCPStdioTool):
    """This class representsfilesystem server with stderr sent to DEVNULL, its working directory set to workspace"""

    def get_mcp_client(self):
        params = StdioServerParameters(command=self.command, args=self.args, env=self.env, cwd=str(workspace))
        return stdio_client(server=params, errlog=subprocess.DEVNULL)


filesystem = FilesystemMCP(
    name="filesystem",
    command="npx",
    args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
)

In [7]:
file_agent = Agent(
    client=client,
    instructions="You can read and write files within your workspace. Use the tools to perform the tasks needed",
    tools=[filesystem],
)

In [8]:
async with file_agent:
    result = await file_agent.run("Read notes.txt and summarize it.")

console.print(Markdown(result.text))

The notes describe a team activity for building a small language tutor in a step-by-step manner. Each “helper”     
selects one task from a shared board, completes it carefully, and marks it as done. The team is instructed to read 
tasks closely, take their time, and do their best.